
# 🧪 Null vs. Alternative: Decision Simulation

This interactive notebook lets students **practice deciding between a null hypothesis (H₀) and an alternative hypothesis (H₁)** using only a **sample statistic** (e.g., a z- or t-statistic) and the **sample size**. 

**How it works (student view):**
1. Click **New Sample**. You’ll see the **calculated test statistic** and **sample size n** (no p-value yet).
2. Decide whether the evidence **supports H₀** or **supports H₁**.
3. After your decision, the notebook reveals:
   - whether the data were truly generated under H₀ or under H₁,
   - whether your decision was correct,
   - and, if incorrect, **what kind of error** it was (**Type I** or **Type II**).
4. Your running stats (correct %, Type I count, Type II count) update each round.

**Instructor controls (at the left):**
- Choose **test type**: proportion test (coin/binary) or mean test.
- Set **H₀**, **direction of H₁**, **effect size** under H₁, **α**, and the **sample size** (fixed or random range).
- Optionally toggle a conceptual **plot** showing the null and alternative distributions **after** a decision is made.

> Tip: Begin with a two-sided test for proportions (e.g., H₀: p = 0.5) with modest n (e.g., 40–100) and α = 0.05.


In [ ]:

import numpy as np
import math
import ipywidgets as W
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

np.random.seed()


In [ ]:

def z_from_proportion(phat, p0, n):
    # classic one-sample proportion z-test
    denom = math.sqrt(p0*(1-p0)/n)
    return (phat - p0) / denom

def z_from_mean(xbar, mu0, sigma, n):
    # z if sigma KNOWN
    return (xbar - mu0) / (sigma / math.sqrt(n))

def t_from_mean(xbar, mu0, s, n):
    # one-sample t using sample standard deviation
    return (xbar - mu0) / (s / math.sqrt(n))

def p_value_from_stat(stat, alternative):
    # two-sided, greater, less under standard normal approximation
    from math import erf
    def Phi(z):
        return 0.5*(1 + erf(z/np.sqrt(2)))
    if alternative == "two-sided":
        return 2*min(Phi(stat), 1-Phi(stat))
    elif alternative == "greater":
        return 1 - Phi(stat)
    else:  # "less"
        return Phi(stat)

def critical_region(alpha, alternative):
    # returns crit function and interval for standard normal
    from math import erfcinv
    def z_quantile(p):
        return -np.sqrt(2)*erfcinv(2*p)
    if alternative == "two-sided":
        zcrit = z_quantile(1 - alpha/2)
        return lambda z: (abs(z) >= zcrit), (-zcrit, zcrit)
    elif alternative == "greater":
        zcrit = z_quantile(1 - alpha)
        return lambda z: (z >= zcrit), (zcrit, np.inf)
    else:  # "less"
        zcrit = z_quantile(alpha)
        return lambda z: (z <= zcrit), (-np.inf, zcrit)


In [ ]:

# ---------- Widgets (Instructor controls) ----------
test_type = W.Dropdown(
    options=[("Proportion (H₀: p = p₀)", "prop"), ("Mean (H₀: μ = μ₀)", "mean")],
    value="prop",
    description="Test:"
)

alt = W.Dropdown(
    options=[("Two-sided", "two-sided"), ("Greater (>)", "greater"), ("Less (<)", "less")],
    value="two-sided",
    description="Alt H₁:"
)

alpha = W.FloatSlider(value=0.05, min=0.005, max=0.20, step=0.005, description="α",
                      readout_format=".3f", continuous_update=False)

# Proportion parameters
p0 = W.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.01, description="p₀",
                   readout_format=".2f", continuous_update=False)
p_effect = W.FloatSlider(value=0.65, min=0.05, max=0.95, step=0.01, description="p (H₁)",
                         readout_format=".2f", continuous_update=False)

# Mean parameters
mu0 = W.FloatText(value=0.0, description="μ₀")
sigma_known = W.Checkbox(value=True, description="Use known σ (z-test)")
sigma = W.FloatText(value=1.0, description="σ (known)")
mu_effect = W.FloatText(value=0.5, description="μ (H₁)")

# Sample size controls
n_fixed = W.IntText(value=60, description="n (fixed)")
n_min = W.IntSlider(value=40, min=10, max=500, step=1, description="n min", continuous_update=False)
n_max = W.IntSlider(value=100, min=10, max=500, step=1, description="n max", continuous_update=False)
randomize_n = W.Checkbox(value=True, description="Randomize n in [min, max]")

# Generation odds
p_H0_true = W.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description="Pr(H₀ true)",
                          readout_format=".2f", continuous_update=False)

# Reveal/plot controls
show_plot_after = W.Checkbox(value=True, description="Plot after decision")
clear_on_new = W.Checkbox(value=True, description="Clear output on New Sample")

controls_left = W.VBox([
    test_type, alt, alpha,
    W.HTML("<hr><b>Proportion test params</b>"),
    p0, p_effect,
    W.HTML("<hr><b>Mean test params</b>"),
    mu0, sigma_known, sigma, mu_effect,
    W.HTML("<hr><b>Sample size</b>"),
    randomize_n, n_min, n_max, n_fixed,
    W.HTML("<hr><b>Generation</b>"),
    p_H0_true,
    W.HTML("<hr><b>Display</b>"),
    show_plot_after, clear_on_new
])

# ---------- Student-facing controls ----------
new_btn = W.Button(description="New Sample", button_style="primary")
support_H0 = W.Button(description="Supports H₀", button_style="")
support_H1 = W.Button(description="Supports H₁", button_style="warning")

info_out = W.Output()
decision_out = W.Output()
plot_out = W.Output()
score_out = W.Output()

buttons = W.HBox([support_H0, support_H1])

# ---------- State ----------
state = {"current": None, "rounds": 0, "correct": 0, "type1": 0, "type2": 0}

def pick_n():
    if randomize_n.value:
        low = min(n_min.value, n_max.value)
        high = max(n_min.value, n_max.value)
        return np.random.randint(low, high+1)
    else:
        return max(2, int(n_fixed.value))

def generate_round():
    # Create a new round based on current settings.
    n = pick_n()
    H0_true = (np.random.rand() < p_H0_true.value)
    if test_type.value == "prop":
        # Bernoulli data
        true_p = p0.value if H0_true else p_effect.value
        x = np.random.binomial(n, true_p)
        phat = x / n
        z = z_from_proportion(phat, p0.value, n)
        stat_label = "z (proportion)"
        extra = {"x": int(x), "phat": phat, "true_p": true_p}
        stat = z
    else:
        # mean test
        if sigma_known.value:
            true_mu = mu0.value if H0_true else mu_effect.value
            data = np.random.normal(loc=true_mu, scale=sigma.value, size=n)
            xbar = data.mean()
            z = z_from_mean(xbar, mu0.value, sigma.value, n)
            stat_label = "z (mean, σ known)"
            extra = {"xbar": xbar, "true_mu": true_mu, "sd_used": sigma.value}
            stat = z
        else:
            true_mu = mu0.value if H0_true else mu_effect.value
            data = np.random.normal(loc=true_mu, scale=sigma.value, size=n)
            xbar = data.mean()
            s = data.std(ddof=1)
            t = t_from_mean(xbar, mu0.value, s, n)
            stat_label = "t (mean, σ unknown)"
            extra = {"xbar": xbar, "s": s, "true_mu": true_mu}
            stat = t

    crit_fn, crit_interval = critical_region(alpha.value, alt.value)
    return {"n": n, "H0_true": H0_true, "stat": stat, "stat_label": stat_label,
            "alt": alt.value, "crit_fn": crit_fn, "crit_interval": crit_interval, "extra": extra}

def describe_round(r):
    # Brief shown to student BEFORE decision.
    s = f"Sample size n = {r['n']}  \n"
    s += f"Calculated statistic: **{r['stat_label']} = {r['stat']:.3f}**  \n"
    direction = {"two-sided":"two-sided", "greater":"right-tailed (>)", "less":"left-tailed (<)"}[r["alt"]]
    s += f"Test direction: **{direction}**  \n"
    s += f"Significance level: **α = {alpha.value:.3f}**"
    return s

def reveal_truth(r, student_choice):
    # Reveal ground truth and evaluate student's decision.
    rejected = r["crit_fn"](r["stat"])
    student_reject = (student_choice == "H1")
    correct = (student_reject == rejected)
    err_type = None
    if not correct:
        if (not r["H0_true"]) and (not student_reject):
            err_type = "Type II (missed a real effect)"
        elif r["H0_true"] and student_reject:
            err_type = "Type I (false positive)"

    state["rounds"] += 1
    if correct:
        state["correct"] += 1
    else:
        if err_type and err_type.startswith("Type I"):
            state["type1"] += 1
        elif err_type and err_type.startswith("Type II"):
            state["type2"] += 1

    dec = "REJECT H₀" if rejected else "FAIL TO REJECT H₀"
    a, b = r["crit_interval"]
    btxt = f"{b:.3f}" if np.isfinite(b) else "∞"
    crit_txt = f"Critical region (approx, z-scale): {a:.3f} to {btxt}"

    msg = []
    msg.append(f"**Your choice:** {'Supports H₁ (reject H₀)' if student_reject else 'Supports H₀ (fail to reject H₀)'}")
    msg.append(f"**Test rule outcome:** {dec}")
    msg.append(f"**Truth for this round:** {'H₀ was TRUE' if r['H0_true'] else 'H₁ was TRUE'}")
    msg.append(f"**Result:** {'✅ Correct!' if correct else '❌ Incorrect.'}")
    if not correct and err_type:
        msg.append(f"**Error type:** {err_type}")

    if test_type.value == "prop":
        msg.append(f"x = {r['extra']['x']} successes out of n = {r['n']} → p̂ = {r['extra']['phat']:.3f}")
        msg.append(f"(Under H₁, true p = {r['extra']['true_p']:.3f})")
    else:
        if 's' in r['extra']:
            msg.append(f"x̄ = {r['extra']['xbar']:.3f}, s = {r['extra']['s']:.3f}")
        else:
            msg.append(f"x̄ = {r['extra']['xbar']:.3f}, σ (known) = {r['extra']['sd_used']:.3f}")
        msg.append(f"(Under H₁, true μ = {r['extra']['true_mu']:.3f})")

    msg.append(crit_txt)
    return "\n\n".join(msg)

def maybe_plot_after(r):
    if not show_plot_after.value:
        return
    # Conceptual null vs alternative on z-scale with observed stat
    xs = np.linspace(-4, 4, 400)
    null_pdf = (1/np.sqrt(2*np.pi))*np.exp(-0.5*xs**2)
    # Illustrative shift for alternative
    if test_type.value == "prop":
        n = r["n"]
        pnull = p0.value
        palt = p_effect.value
        denom = np.sqrt(pnull*(1-pnull)/n)
        shift = (palt - pnull)/denom if denom > 0 else 0.0
    else:
        n = r["n"]
        denom = sigma.value/np.sqrt(n)
        shift = (mu_effect.value - mu0.value)/denom if denom > 0 else 0.0

    alt_pdf = (1/np.sqrt(2*np.pi))*np.exp(-0.5*(xs-shift)**2)

    plt.figure(figsize=(6,4))
    plt.plot(xs, null_pdf, label="Null (≈ N(0,1))")
    plt.plot(xs, alt_pdf, label="Alternative (shifted)")
    a, b = r["crit_interval"]
    if np.isfinite(a):
        plt.axvline(a, linestyle="--", label="critical boundary")
    if np.isfinite(b):
        plt.axvline(b, linestyle="--")
    plt.axvline(r["stat"], linewidth=3, label=f"observed {r['stat_label'].split()[0]}")
    plt.xlabel("Test statistic scale (≈ z)")
    plt.ylabel("Density (conceptual)")
    plt.title("Conceptual view of null vs. alternative (not exact)")
    plt.legend()
    plt.show()

def update_scoreboard():
    with score_out:
        clear_output(wait=True)
        acc = 0.0 if state["rounds"] == 0 else 100.0 * state["correct"] / state["rounds"]
        display(W.HTML(
            f"<b>Rounds:</b> {state['rounds']} &nbsp; | &nbsp; "
            f"<b>Correct:</b> {state['correct']} ({acc:.1f}%) &nbsp; | &nbsp; "
            f"<b>Type I:</b> {state['type1']} &nbsp; | &nbsp; "
            f"<b>Type II:</b> {state['type2']}"
        ))

def on_new_clicked(_):
    if clear_on_new.value:
        info_out.clear_output()
        decision_out.clear_output()
        plot_out.clear_output()
    r = generate_round()
    state['current'] = r
    with info_out:
        clear_output(wait=True)
        print(describe_round(r))

def on_support_H0(_):
    decide("H0")

def on_support_H1(_):
    decide("H1")

def decide(choice):
    r = state["current"]
    if r is None:
        with decision_out:
            clear_output(wait=True)
            print("Click 'New Sample' first.")
        return
    with decision_out:
        clear_output(wait=True)
        print(reveal_truth(r, choice))
    with plot_out:
        clear_output(wait=True)
        maybe_plot_after(r)
    update_scoreboard()

new_btn.on_click(on_new_clicked)
support_H0.on_click(on_support_H0)
support_H1.on_click(on_support_H1)

# Initial scoreboard
update_scoreboard()

ui = W.HBox([controls_left, W.VBox([new_btn, info_out, buttons, decision_out, plot_out, score_out])])
ui
